# Système de recommandation agricole - Modélisation - premiers pas

In [1]:
# Import de base
import pandas as pd
import numpy as np
import sys,os
sys.path.append(os.path.abspath(".."))

# Import scikit learn
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.dummy import DummyRegressor

# Import des autres modèles testés
import xgboost as xgb
import lightgbm as lgb
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

# Import mlflow
import mlflow
import mlflow.sklearn
mlflow.set_tracking_uri("file://" + os.path.abspath("../mlruns"))
from mlflow.tracking import MlflowClient

# Import du projet
from scripts.preprocessing_pipeline import (
    separation_X_y,
    preparation_pipeline,
    cross_validation,
    train_predict
)

In [ ]:
def mlflow_tracking_model(model,model_name,tags,projet_description):
    # ==========================================
    # Configuration MLflow
    mlflow.set_tracking_uri("http://127.0.0.1:5000")
    mlflow.set_experiment("Yield_Prediction_Agricultural")

    # L'autolog va capturer automatiquement les paramètres du modèle
    mlflow.sklearn.autolog(log_models=True, log_datasets=False, silent=True)

    # ==========================================
    # Chargement des données
    df = pd.read_csv("../data/processed/yield_df_final_conso_encoded.csv", index_col=0)
    model_name
    tags
    projet_description


    with mlflow.start_run(run_name=model_name, tags={"Training Info" : tags,
                            "mlflow.note.content" : projet_description
                            }):

        # Logs infos générales (Metadata)
        mlflow.log_params({
            "dataset": "yield_df_final_conso_encoded.csv",
            "n_rows": df.shape[0],
            "n_cols": df.shape[1],
            "random_state": 42
        })

        # Separation
        X_train, X_test, y_train, y_test, categorical_cols, numeric_cols = separation_X_y(df)

        # ==========================================
        # Modèle et Pipeline
        model = model
        pipeline = preparation_pipeline(
            numeric_cols=numeric_cols,
            categorical_cols=categorical_cols,
            model=model
        )

        # ==========================================
        # Cross-validation
        cv_results = cross_validation(
            pipeline=pipeline,
            X_train=X_train,
            y_train=y_train
        )
        
        # Calcul des scores CV
        cv_metrics = {
            "cv_rmse": np.sqrt(-cv_results["test_mse"]).mean(),
            "cv_mae": (-cv_results["test_mae"]).mean(),
            "cv_r2": cv_results["test_r2"].mean()
        }
        mlflow.log_metrics(cv_metrics)

        # ==========================================
        # Entraînement Final (L'Autolog capture tout ici)
        pipeline.fit(X_train, y_train)

        # ==========================================
        # Prédictions et Test Metrics
        y_pred = pipeline.predict(X_test)
        # y_test = valeurs réelles, y_pred = valeurs prédites
        erreurs_relatives = abs(y_test - y_pred) / y_test
        precision_eco = (erreurs_relatives <= 0.10).mean() * 100

        print(f"Précision Économique : {precision_eco:.2f}%")

        test_metrics = {
            "test_rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
            "test_mae": mean_absolute_error(y_test, y_pred),
            "test_r2": r2_score(y_test, y_pred),
            "Précision Économique" : precision_eco
        }
        mlflow.log_metrics(test_metrics)

        # mlflow.sklearn.log_model(pipeline, "model", pip_requirements=["scikit-learn", "pandas", "numpy"])

        print("\n=== RandomForest avec Autolog & CV ===")
        print(f"CV RMSE   : {cv_metrics['cv_rmse']:.4f}")
        print(f"Test RMSE : {test_metrics['test_rmse']:.4f}")
        print(f"Test R2   : {test_metrics['test_r2']:.4f}")

In [25]:
model = RandomForestRegressor(random_state=42)
model_name = "RandomForest"
tags = "RandomForest"
projet_description = "Premier test de description"
mlflow_tracking_model(model, model_name, tags, projet_description)


2026-03-30 18:08:19,096 - INFO - ['avg_temp', 'rainfall_mm', 'pesticides_tonnes', 'tech_trend', 'climate_instability', 'relative_tech_intensity']
2026-03-30 18:08:19,096 - INFO - ['region_australia_and_new_zealand', 'region_central_asia', 'region_eastern_asia', 'region_eastern_europe', 'region_latin_america_and_the_caribbean', 'region_melanesia', 'region_micronesia', 'region_northern_africa', 'region_northern_america', 'region_northern_europe', 'region_polynesia', 'region_south-eastern_asia', 'region_southern_asia', 'region_southern_europe', 'region_sub-saharan_africa', 'region_western_asia', 'region_western_europe', 'item_cassava', 'item_maize', 'item_plantains_and_others', 'item_potatoes', 'item_rice', 'item_sorghum', 'item_soybean', 'item_sweet_potatoes', 'item_wheat', 'item_yams']



=== RandomForest ===
R2 Test           : 0.9439
Précision Stricte : 60.47%
Précision Hybride : 75.97% (Cible Métier)
🏃 View run RandomForest at: http://127.0.0.1:5000/#/experiments/1/runs/69026dca4d8c4c9ead9f20371e046486
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [18]:
model = LinearRegression()
model_name = "LinearRegression"
mlflow_tracking_model(model, model_name)

2026-03-30 17:35:36,641 - INFO - ['avg_temp', 'rainfall_mm', 'pesticides_tonnes', 'tech_trend', 'climate_instability', 'relative_tech_intensity']
2026-03-30 17:35:36,642 - INFO - ['region_australia_and_new_zealand', 'region_central_asia', 'region_eastern_asia', 'region_eastern_europe', 'region_latin_america_and_the_caribbean', 'region_melanesia', 'region_micronesia', 'region_northern_africa', 'region_northern_america', 'region_northern_europe', 'region_polynesia', 'region_south-eastern_asia', 'region_southern_asia', 'region_southern_europe', 'region_sub-saharan_africa', 'region_western_asia', 'region_western_europe', 'item_cassava', 'item_maize', 'item_plantains_and_others', 'item_potatoes', 'item_rice', 'item_sorghum', 'item_soybean', 'item_sweet_potatoes', 'item_wheat', 'item_yams']



=== RandomForest avec Autolog & CV ===
CV RMSE   : 48452.9898
Test RMSE : 48180.7848
Test R2   : 0.5740
🏃 View run LinearRegression at: http://127.0.0.1:5000/#/experiments/1/runs/02c919a3ba6942fe8f277cd71e14e75b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
